# PyGeoVision InSAR Processing Chain

End-to-end InSAR from co-registered SAR rasters to interpreted deformation report.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').parent))
from pygeovision.insar import InSARProcessor, InSARViz, InSARInterpreter
from pygeovision.insar.interferogram import generate_interferogram, amplitude_coherence
from pygeovision.insar.coherence import estimate_coherence, coherence_mask
from pygeovision.insar.displacement import phase_to_displacement, displacement_rate
print("PyGeoVision InSAR chain loaded")

## High-level API: `InSARProcessor`

In [ ]:
# Chained API — same pattern as PyGeoVision client
proc = InSARProcessor(
    output_dir="./results/insar/",
    wavelength=0.05546576,      # Sentinel-1 C-band (metres)
    incidence_angle_deg=39.0,   # typical IW mode incidence angle
    coherence_threshold=0.4,    # mask below this in displacement maps
)
print(proc)
print()
print("Usage:")
print("  result = proc.interferogram('pre.tif', 'post.tif')")
print("  result = proc.displacement(result)")
print("  rate   = proc.deformation_rate(['d0.tif','d1.tif'], ['2023-01-01','2024-01-01'])")
print("  full   = proc.full_pipeline('pre.tif', 'post.tif', study_area='Istanbul')")

## Interferogram generation

In [ ]:
# generate_interferogram: GRD amplitude change proxy wrapped to [-π, +π]
# In GRD mode (no SLC required):
#   1. Amplitude change ratio = (post-pre)/(post+pre)
#   2. Wrapped to [-π, π] for visual consistency with true InSAR
#   3. Goldstein adaptive phase filter applied
#   4. Amplitude coherence proxy (0=decorrelated, 1=coherent)
#
# For true SLC InSAR (mm precision):
#   → pyroSAR + ESA SNAP → unwrapped phase → import here
#
# result = generate_interferogram(
#     pre_path="sar_pre_ready.tif",
#     post_path="sar_post_ready.tif",
#     output_dir="./insar/",
#     multilook=2,              # reduces noise, reduces resolution
#     filter_type="goldstein",  # adaptive phase filter
# )
# print(f"Interferogram: {result['interferogram_path']}")
# print(f"Coherence:     {result['coherence_path']}")
# print(f"Mean coherence: {result['stats']['mean_coherence']:.3f}")
# print(f"Changed area:   {result['stats']['changed_pct']:.1f}%")
print("Interferogram generation: see generate_interferogram() in pygeovision.insar.interferogram")
print("Key parameters: multilook, filter_type (goldstein/gaussian/none)")

## Phase → Displacement

In [ ]:
# Convert wrapped phase / amplitude change to displacement map
# 
# SLC mode (true InSAR):   d_LOS [m] = λ/(4π) × φ_unwrapped [rad]
# GRD amplitude proxy:     d_proxy [m] = change_ratio × 0.5
#
# disp_m = phase_to_displacement(
#     "interferogram.tif",
#     output_path="displacement.tif",
#     mode="amplitude_proxy",       # or "slc_phase"
#     wavelength=0.05546576,        # Sentinel-1
#     incidence_angle_deg=39.0,
#     vertical_only=False,          # True → project LOS to vertical
# )
print("Displacement conversion modes:")
print("  'slc_phase'       → d_LOS = λ/(4π) × φ_unwrapped  [mm precision, SLC required]")
print("  'amplitude_proxy' → empirical scaling of GRD change ratio  [cm-dm accuracy]")

## Deformation rate (time-series)

In [ ]:
# Pixel-wise linear regression: displacement stack → mm/year
# Uses decimal year timestamps from ISO date strings
# 
# rate = displacement_rate(
#     displacement_paths=["disp_jan23.tif", "disp_jul23.tif", "disp_jan24.tif"],
#     dates=["2023-01-01", "2023-07-01", "2024-01-01"],
#     output_path="deformation_rate_mm_yr.tif",
#     mask_path="coherence.tif",   # optional — masks low-coherence pixels
# )
# print(f"Rate range: {rate.min():.1f} to {rate.max():.1f} mm/yr")
print("Deformation rate: pixel-wise linear regression across displacement time-series")
print("Output: mm/year (negative = subsidence, positive = uplift)")

## Interpretation: `InSARInterpreter`

In [ ]:
# Auto-detect zones, generate flags, export structured report
# 
# interp = InSARInterpreter(
#     subsidence_threshold_m=-0.02,  # -2 cm → subsidence zone
#     uplift_threshold_m=+0.01,      # +1 cm → uplift zone
#     coherence_threshold=0.4,
#     min_zone_km2=0.1,
# )
# report = interp.interpret(
#     "displacement.tif",
#     rate_path="rate.tif",           # optional
#     coherence_path="coherence.tif", # optional
#     study_area="Istanbul, Turkey",
# )
# print(report.summary())
# report.export("insar_report.json")
# report.export("insar_report.md")
#
from pygeovision.insar.interpretation import InSARInterpreter, DeformationReport, DeformationZone
print("InSARInterpreter → DeformationReport")
print("Report fields: max_subsidence_m, max_uplift_m, zones[], flags[]")
print("Export formats: JSON, Markdown")

## Visualization: `InSARViz`

In [ ]:
from pygeovision.insar.visualization import InSARViz
viz = InSARViz(figsize=(12, 9))
print("InSARViz methods:")
for m in sorted([m for m in dir(InSARViz) if not m.startswith('_')]):
    print(f"  viz.{m}(...)")
print()
print("Dashboard example:")
print("  viz.dashboard(")
print("      interferogram_path='ifg.tif',")
print("      coherence_path='coh.tif',")
print("      displacement_path='disp.tif',")
print("      rate_path='rate_mm_yr.tif',")
print("  ).export('insar_dashboard.png')")

## Summary — InSAR chain

```
pre_sar.tif + post_sar.tif
    │
    ▼  generate_interferogram()
 interferogram.tif  +  coherence.tif
    │
    ▼  phase_to_displacement()
 displacement.tif  [metres LOS]
    │
    ▼  displacement_rate() [time-series]
 deformation_rate_mm_yr.tif
    │
    ▼  InSARInterpreter.interpret()
 DeformationReport → JSON / Markdown
```

**For mm-precision:** process Sentinel-1 SLC → pyroSAR + SNAP → unwrapped phase → import here.